# Diapilot — Experiment 4: Random Forest Regressor
### Ensemble Learning with Feature Importance Analysis
**Key advantage over other models:** Random Forest provides feature importance scores — showing WHICH nutritional factors matter most for diabetic meal recommendations. This is the only model in the pipeline that explains its own decisions.

In [1]:
# ════════════════════════════════════════════════════════
# CELL 1 — IMPORTS & SETUP
# ════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import os

print('=' * 58)
print('   DIAPILOT — Experiment 4: Random Forest Regressor')
print('=' * 58)

df = pd.read_parquet('../data/processed/DiaPilot_Combined_Data.parquet')
print(f'Dataset loaded: {len(df):,} recipes')

   DIAPILOT — Experiment 4: Random Forest Regressor
Dataset loaded: 117,662 recipes


In [2]:
# ════════════════════════════════════════════════════════
# CELL 2 — MEDICAL ENGINE (ADA-ALIGNED)
# Identical to KNN, XGBoost, DNN for fair comparison
# ════════════════════════════════════════════════════════

def calculate_tdee(age, weight_kg, height_cm, gender, activity='sedentary'):
    """Mifflin-St Jeor Equation."""
    if gender.lower() == 'male':
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) + 5
    else:
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) - 161
    multipliers = {'sedentary':1.2, 'light':1.375,
                   'moderate':1.55, 'active':1.725}
    return round(bmr * multipliers.get(activity, 1.2))


def get_dynamic_macros(tdee, leicester_score, on_medication=False,
                        current_glucose_high=False):
    """ADA 2025-aligned per-meal constraints."""
    if leicester_score <= 10:
        carb_pct, sugar_limit = 0.50, 15.0
    elif leicester_score <= 15:
        carb_pct, sugar_limit = 0.40, 10.0
    else:
        carb_pct, sugar_limit = 0.30,  8.0

    if current_glucose_high:
        carb_pct   -= 0.10
        sugar_limit =  3.0

    per_meal_carbs_max = round(((tdee * carb_pct) / 4) / 3)
    per_meal_carbs_min = 25 if on_medication else 0
    if on_medication and per_meal_carbs_max < 25:
        per_meal_carbs_max = 25

    return {
        'per_meal_calories'  : round(tdee / 3),
        'per_meal_max_carbs' : per_meal_carbs_max,
        'per_meal_min_carbs' : per_meal_carbs_min,
        'per_meal_sugar'     : sugar_limit
    }


# ── Patient profile ─────────────────────────────────────
PATIENT_AGE       = 45
PATIENT_WEIGHT_KG = 85
PATIENT_HEIGHT_CM = 175
PATIENT_GENDER    = 'male'
PATIENT_ACTIVITY  = 'sedentary'
LEICESTER_SCORE   = 14
ON_MEDICATION     = True
GLUCOSE_HIGH      = False

patient_tdee   = calculate_tdee(PATIENT_AGE, PATIENT_WEIGHT_KG,
                                 PATIENT_HEIGHT_CM, PATIENT_GENDER,
                                 PATIENT_ACTIVITY)
patient_macros = get_dynamic_macros(patient_tdee, LEICESTER_SCORE,
                                     ON_MEDICATION, GLUCOSE_HIGH)

print('Medical Engine Ready')
print('-' * 40)
print(f'  TDEE             : {patient_tdee} kcal/day')
print(f'  Per-meal calories: {patient_macros["per_meal_calories"]} kcal')
print(f'  Carb window      : {patient_macros["per_meal_min_carbs"]}g'
      f' - {patient_macros["per_meal_max_carbs"]}g')
print(f'  Sugar max        : {patient_macros["per_meal_sugar"]}g')

Medical Engine Ready
----------------------------------------
  TDEE             : 2068 kcal/day
  Per-meal calories: 689 kcal
  Carb window      : 25g - 69g
  Sugar max        : 10.0g


In [3]:
# ════════════════════════════════════════════════════════
# CELL 3 — SYNTHETIC TRAINING DATA GENERATION
# Same approach as DNN — multiple patient profiles
# so RF learns patient-specific relevance patterns
# ════════════════════════════════════════════════════════

def calculate_meal_relevance(meal_row, macros):
    """
    Clinically grounded relevance score (0.0 - 1.0).
    Returns 0.0 if meal violates any hard medical constraint.

    Weights:
      Carb proximity  50% — closeness to patient's ideal carb target
      Protein reward  30% — higher protein stabilizes blood sugar
      Sugar penalty   20% — lower sugar = better glycaemic control
    """
    # Hard constraint — fail fast
    if (meal_row['Calories']  > macros['per_meal_calories']  or
        meal_row['Carbs_g']   > macros['per_meal_max_carbs'] or
        meal_row['Carbs_g']   < macros['per_meal_min_carbs'] or
        meal_row['Sugar_g']   > macros['per_meal_sugar']):
        return 0.0

    carb_target  = (macros['per_meal_max_carbs'] +
                    macros['per_meal_min_carbs']) / 2
    carb_range   = max(macros['per_meal_max_carbs'] -
                       macros['per_meal_min_carbs'], 1)
    carb_score   = max(0.0, 1.0 - abs(meal_row['Carbs_g'] -
                                       carb_target) / carb_range)

    protein_score = min(1.0, meal_row['Protein_g'] / 50.0)

    sugar_score  = max(0.0, 1.0 - meal_row['Sugar_g'] /
                       (macros['per_meal_sugar'] + 1e-9))

    return round(
        carb_score    * 0.50 +
        protein_score * 0.30 +
        sugar_score   * 0.20,
        4
    )


# Same 10 patient profiles as DNN for consistent comparison
SYNTHETIC_PATIENTS = [
    (25, 60, 165, 'female', 'moderate',   6, False),
    (35, 75, 170, 'male',   'light',      9, False),
    (42, 80, 168, 'female', 'sedentary', 12, False),
    (45, 85, 175, 'male',   'sedentary', 14, True ),
    (50, 90, 172, 'male',   'sedentary', 16, True ),
    (55, 70, 160, 'female', 'light',     15, False),
    (60, 95, 178, 'male',   'sedentary', 18, True ),
    (38, 65, 162, 'female', 'moderate',  11, False),
    (48, 88, 174, 'male',   'light',     17, True ),
    (30, 55, 158, 'female', 'active',     5, False),
]

print('Generating synthetic training pairs...')
print(f'  Patient profiles  : {len(SYNTHETIC_PATIENTS)}')
print(f'  Recipes available : {len(df):,}')

training_rows = []

for (age, wt, ht, gen, act, leic, med) in SYNTHETIC_PATIENTS:
    tdee   = calculate_tdee(age, wt, ht, gen, act)
    macros = get_dynamic_macros(tdee, leic, med)
    bmi    = round(wt / ((ht / 100) ** 2), 1)

    # Include meals slightly outside constraints so RF
    # learns the boundary — not just the easy cases
    candidate_df = df[
        (df['Calories'] <= macros['per_meal_calories'] * 1.2) &
        (df['Carbs_g']  <= macros['per_meal_max_carbs'] * 1.3) &
        (df['Sugar_g']  <= macros['per_meal_sugar'] * 1.5)
    ]

    for _, meal in candidate_df.iterrows():
        score = calculate_meal_relevance(meal, macros)
        training_rows.append({
            # Patient features
            'p_age'         : age,
            'p_bmi'         : bmi,
            'p_leicester'   : leic,
            'p_tdee'        : tdee,
            'p_medication'  : int(med),
            'p_carb_target' : (macros['per_meal_max_carbs'] +
                               macros['per_meal_min_carbs']) / 2,
            'p_sugar_limit' : macros['per_meal_sugar'],
            'p_cal_limit'   : macros['per_meal_calories'],
            # Meal features
            'm_calories'    : meal['Calories'],
            'm_carbs'       : meal['Carbs_g'],
            'm_sugar'       : meal['Sugar_g'],
            'm_protein'     : meal['Protein_g'],
            'm_fat'         : meal['Fat_g'],
            # Target
            'relevance'     : score
        })

train_df = pd.DataFrame(training_rows)
print(f'\nTraining pairs generated : {len(train_df):,}')
print(f'  Positive (score > 0)   : {(train_df["relevance"] > 0).sum():,}')
print(f'  Zero-score (violations): {(train_df["relevance"] == 0).sum():,}')
print(f'  Avg relevance score    : {train_df["relevance"].mean():.4f}')

Generating synthetic training pairs...
  Patient profiles  : 10
  Recipes available : 117,662

Training pairs generated : 984,759
  Positive (score > 0)   : 569,510
  Zero-score (violations): 415,249
  Avg relevance score    : 0.3319


In [4]:
# ════════════════════════════════════════════════════════
# CELL 4 — FEATURE PREPARATION
# ════════════════════════════════════════════════════════

FEATURE_COLS = [
    # Patient features
    'p_age', 'p_bmi', 'p_leicester', 'p_tdee',
    'p_medication', 'p_carb_target', 'p_sugar_limit', 'p_cal_limit',
    # Meal features
    'm_calories', 'm_carbs', 'm_sugar', 'm_protein', 'm_fat'
]

# Human-readable names for feature importance chart
FEATURE_NAMES = [
    'Patient Age', 'Patient BMI', 'Leicester Score', 'Patient TDEE',
    'On Medication', 'Carb Target', 'Sugar Limit', 'Calorie Limit',
    'Meal Calories', 'Meal Carbs', 'Meal Sugar', 'Meal Protein', 'Meal Fat'
]

X = train_df[FEATURE_COLS].values.astype(np.float32)
y = train_df['relevance'].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

# Random Forest is scale-invariant — no normalization needed
# This is one advantage over DNN and XGBoost
print('Feature preparation complete')
print('-' * 40)
print(f'  Total features   : {len(FEATURE_COLS)}')
print(f'    Patient inputs : 8')
print(f'    Meal inputs    : 5')
print(f'  Training samples : {len(X_train):,}')
print(f'  Test samples     : {len(X_test):,}')
print()
print('Note: Random Forest requires NO feature normalization.')
print('Trees split on raw values, making it scale-invariant.')

Feature preparation complete
----------------------------------------
  Total features   : 13
    Patient inputs : 8
    Meal inputs    : 5
  Training samples : 837,045
  Test samples     : 147,714

Note: Random Forest requires NO feature normalization.
Trees split on raw values, making it scale-invariant.


In [5]:
# ════════════════════════════════════════════════════════
# CELL 5 — RANDOM FOREST TRAINING
#
# Hyperparameter choices explained:
#   n_estimators=300  : 300 trees — enough for stable predictions
#                       without excessive training time
#   max_depth=12      : prevents overfitting on clinical labels
#                       while capturing complex patient-meal interactions
#   min_samples_leaf=5: each leaf needs 5+ samples — prevents
#                       memorizing individual outlier meals
#   max_features='sqrt': each tree sees sqrt(13)~3-4 features
#                        forces diversity across trees
#   n_jobs=-1         : uses all CPU cores for parallel training
# ════════════════════════════════════════════════════════

print('Training Random Forest...')
print('(300 trees — may take 1-2 minutes)')
print('-' * 40)

rf_model = RandomForestRegressor(
    n_estimators    = 300,
    max_depth       = 12,
    min_samples_leaf= 5,
    max_features    = 'sqrt',
    random_state    = 42,
    n_jobs          = -1,
    verbose         = 0
)

rf_model.fit(X_train, y_train)

print('Random Forest training complete.')
print(f'  Trees trained    : {rf_model.n_estimators}')
print(f'  Max depth used   : {rf_model.max_depth}')
print(f'  Features per tree: sqrt({len(FEATURE_COLS)}) = '
      f'{int(len(FEATURE_COLS)**0.5)}')

Training Random Forest...
(300 trees — may take 1-2 minutes)
----------------------------------------
Random Forest training complete.
  Trees trained    : 300
  Max depth used   : 12
  Features per tree: sqrt(13) = 3


In [6]:
# ════════════════════════════════════════════════════════
# CELL 6 — MODEL EVALUATION
# ════════════════════════════════════════════════════════

y_pred_train = rf_model.predict(X_train)
y_pred_test  = rf_model.predict(X_test)

# Core metrics
train_mae  = mean_absolute_error(y_train, y_pred_train)
test_mae   = mean_absolute_error(y_test,  y_pred_test)
train_r2   = r2_score(y_train, y_pred_train)
test_r2    = r2_score(y_test,  y_pred_test)
test_rmse  = np.sqrt(np.mean((y_test - y_pred_test) ** 2))

# Overfitting check: gap between train and test R2
overfit_gap = train_r2 - test_r2

# Recommendation quality
threshold   = 0.3
recommended = y_pred_test >= threshold
actual_good = y_test       > 0
precision   = (recommended & actual_good).sum() / max(recommended.sum(), 1)
recall      = (recommended & actual_good).sum() / max(actual_good.sum(),  1)
f1          = 2 * precision * recall / max(precision + recall, 1e-9)

# 5-fold cross validation for robust estimate
print('Running 5-fold cross validation...')
cv_scores = cross_val_score(
    rf_model, X, y,
    cv=5, scoring='r2', n_jobs=-1
)

print('=' * 58)
print('   RANDOM FOREST EVALUATION REPORT')
print('=' * 58)

print(f'\n1. Regression Metrics')
print(f'   Train MAE  : {train_mae:.4f}  |  Test MAE  : {test_mae:.4f}')
print(f'   Train R2   : {train_r2:.4f}  |  Test R2   : {test_r2:.4f}')
print(f'   Test RMSE  : {test_rmse:.4f}')
print(f'   Overfit gap: {overfit_gap:.4f}  '
      f'({"acceptable" if overfit_gap < 0.1 else "overfitting — reduce depth"})')

print(f'\n2. Cross-Validation R2 (5-fold)')
print(f'   Scores     : {[round(s,4) for s in cv_scores]}')
print(f'   Mean       : {cv_scores.mean():.4f}')
print(f'   Std        : {cv_scores.std():.4f}  '
      f'(lower = more stable)')

print(f'\n3. Recommendation Quality (threshold={threshold})')
print(f'   Precision  : {precision:.4f}')
print(f'   Recall     : {recall:.4f}')
print(f'   F1 Score   : {f1:.4f}')

print('=' * 58)

Running 5-fold cross validation...


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — FEATURE IMPORTANCE ANALYSIS
#
# This is the unique contribution of Random Forest.
# No other model in this pipeline can tell you WHICH
# features matter most for making good recommendations.
#
# For your panel: this directly answers the clinical
# question of "what does the AI consider most important
# when recommending meals for a diabetic patient?"
# ════════════════════════════════════════════════════════

importances = rf_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature'   : FEATURE_NAMES,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

importance_df['Importance_Pct'] = (
    importance_df['Importance'] * 100
).round(2)

importance_df['Type'] = importance_df['Feature'].apply(
    lambda x: 'Patient' if x.startswith('Patient') or
    x in ['Leicester Score','On Medication',
           'Carb Target','Sugar Limit','Calorie Limit']
    else 'Meal'
)

print('=' * 58)
print('   FEATURE IMPORTANCE ANALYSIS')
print('   (What the AI considers most important)')
print('=' * 58)
print()

for _, row in importance_df.iterrows():
    bar   = '|' * int(row['Importance_Pct'] * 1.5)
    label = f'[{row["Type"][:3]}]'
    print(f'  {label} {row["Feature"]:22s}: '
          f'{row["Importance_Pct"]:5.2f}%  {bar}')

print()
# Summarize patient vs meal importance
patient_imp = importance_df[
    importance_df['Type'] == 'Patient'
]['Importance'].sum() * 100
meal_imp    = importance_df[
    importance_df['Type'] == 'Meal'
]['Importance'].sum() * 100

print(f'  Patient features total : {patient_imp:.1f}%')
print(f'  Meal features total    : {meal_imp:.1f}%')
print()
top3 = importance_df.head(3)
print('  Top 3 most important factors:')
for i, row in top3.iterrows():
    print(f'    {i+1}. {row["Feature"]} ({row["Importance_Pct"]}%)')
print('=' * 58)

   FEATURE IMPORTANCE ANALYSIS
   (What the AI considers most important)

  [Mea] Meal Sugar            : 32.85%  |||||||||||||||||||||||||||||||||||||||||||||||||
  [Mea] Meal Carbs            : 17.00%  |||||||||||||||||||||||||
  [Pat] On Medication         : 10.41%  |||||||||||||||
  [Mea] Meal Calories         :  7.45%  |||||||||||
  [Pat] Leicester Score       :  7.13%  ||||||||||
  [Pat] Patient Age           :  7.06%  ||||||||||
  [Pat] Patient BMI           :  5.03%  |||||||
  [Pat] Sugar Limit           :  3.67%  |||||
  [Pat] Calorie Limit         :  2.05%  |||
  [Pat] Patient TDEE          :  2.02%  |||
  [Mea] Meal Protein          :  1.93%  ||
  [Pat] Carb Target           :  1.74%  ||
  [Mea] Meal Fat              :  1.65%  ||

  Patient features total : 39.1%
  Meal features total    : 60.9%

  Top 3 most important factors:
    1. Meal Sugar (32.85%)
    2. Meal Carbs (17.0%)
    3. On Medication (10.41%)


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — MEAL TYPE CLASSIFICATION
# ════════════════════════════════════════════════════════

BREAKFAST_KW = ['egg','oat','oatmeal','paratha','halwa','dahi','yogurt',
                'pancake','toast','cereal','porridge','smoothie','muffin',
                'waffle','granola','scramble','omelette','crepe']
DINNER_KW    = ['karahi','biryani','gosht','daal','curry','sabzi','stew',
                'roast','steak','bake','casserole','kebab','tikka','nihari',
                'korma','palak','keema','chops','grilled chicken','lamb']

def classify_meal_type(row):
    text    = (str(row['Recipe_Name']) + ' ' +
               str(row['Ingredients'])).lower()
    b_score = sum(1 for kw in BREAKFAST_KW if kw in text)
    d_score = sum(1 for kw in DINNER_KW   if kw in text)
    if b_score > d_score: return 'Breakfast'
    elif d_score > 0:     return 'Dinner'
    return 'Lunch'

df['Meal_Type'] = df.apply(classify_meal_type, axis=1)
counts = df['Meal_Type'].value_counts()
print('Meal type classification complete')
for mt in ['Breakfast', 'Lunch', 'Dinner']:
    print(f'  {mt:12s}: {counts.get(mt, 0):,} recipes')

Meal type classification complete
  Breakfast   : 28,429 recipes
  Lunch       : 63,925 recipes
  Dinner      : 25,308 recipes


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 9 — INFERENCE: SCORE ALL SAFE MEALS
# ════════════════════════════════════════════════════════

# Medical filter
safe_df = df[
    (df['Calories'] <= patient_macros['per_meal_calories']) &
    (df['Carbs_g']  <= patient_macros['per_meal_max_carbs']) &
    (df['Carbs_g']  >= patient_macros['per_meal_min_carbs']) &
    (df['Sugar_g']  <= patient_macros['per_meal_sugar'])
].copy().reset_index(drop=True)

print(f'Medically safe pool: {len(safe_df):,} recipes')

# Build inference matrix
patient_bmi = PATIENT_WEIGHT_KG / ((PATIENT_HEIGHT_CM / 100) ** 2)

patient_row = np.array([
    PATIENT_AGE,
    patient_bmi,
    LEICESTER_SCORE,
    patient_tdee,
    int(ON_MEDICATION),
    (patient_macros['per_meal_max_carbs'] +
     patient_macros['per_meal_min_carbs']) / 2,
    patient_macros['per_meal_sugar'],
    patient_macros['per_meal_calories']
])

# Repeat patient row for every meal
patient_matrix = np.tile(patient_row, (len(safe_df), 1))
meal_matrix    = safe_df[['Calories','Carbs_g','Sugar_g',
                            'Protein_g','Fat_g']].values

X_inference = np.hstack([patient_matrix, meal_matrix]).astype(np.float32)

# RF does not need scaling — predict directly
print('Running Random Forest inference...')
safe_df['RF_Score'] = rf_model.predict(X_inference)

print(f'Inference complete.')
print(f'  Score range: {safe_df["RF_Score"].min():.4f}'
      f' - {safe_df["RF_Score"].max():.4f}')
print(f'  Avg score  : {safe_df["RF_Score"].mean():.4f}')

Medically safe pool: 21,763 recipes
Running Random Forest inference...
Inference complete.
  Score range: 0.3033 - 0.7077
  Avg score  : 0.5309


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 10 — MMR DIVERSITY SELECTION
# Identical algorithm to KNN, XGBoost, DNN
# ════════════════════════════════════════════════════════

def apply_mmr(candidate_df, score_col, top_k, lambda_param=0.7):
    """
    Generic MMR — works with any score column.
    Diversity measured across 4 macros (Calories, Carbs, Sugar, Protein).
    """
    if len(candidate_df) == 0:
        return pd.DataFrame()
    if len(candidate_df) <= top_k:
        return candidate_df.reset_index(drop=True)

    macro_cols = ['Calories','Carbs_g','Sugar_g','Protein_g']
    macro_vals = candidate_df[macro_cols].values.astype(float)
    col_range  = macro_vals.max(axis=0) - macro_vals.min(axis=0)
    col_range  = np.where(col_range > 0, col_range, 1.0)
    macro_norm = (macro_vals - macro_vals.min(axis=0)) / col_range

    scores    = candidate_df[score_col].values
    remaining = list(range(len(candidate_df)))
    selected  = []

    best_start = int(np.argmax(scores))
    selected.append(best_start)
    remaining.remove(best_start)

    while len(selected) < top_k and remaining:
        best_mmr = -float('inf')
        best_idx = None
        for i in remaining:
            relevance = scores[i]
            sims      = [
                1.0 / (1.0 + np.linalg.norm(
                    macro_norm[i] - macro_norm[s]))
                for s in selected
            ]
            mmr = lambda_param * relevance - \
                  (1 - lambda_param) * max(sims)
            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = i
        selected.append(best_idx)
        remaining.remove(best_idx)

    return candidate_df.iloc[selected].reset_index(drop=True)


print('MMR function ready.')

MMR function ready.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 11 — GENERATE 30-DAY PLAN
# ════════════════════════════════════════════════════════

DAYS_IN_PLAN = 30
MEAL_SLOTS   = ['Breakfast', 'Lunch', 'Dinner']

plan_rows    = []
pool_summary = {}

for meal_type in MEAL_SLOTS:
    pool = safe_df[safe_df['Meal_Type'] == meal_type].copy()

    if len(pool) < 10:
        print(f'  WARNING: {meal_type} only {len(pool)} recipes'
              f' — using full safe pool')
        pool = safe_df.copy()

    selected = apply_mmr(pool, score_col='RF_Score',
                          top_k=DAYS_IN_PLAN)
    pool_summary[meal_type] = len(selected)

    for day_idx in range(DAYS_IN_PLAN):
        row = selected.iloc[day_idx % len(selected)]
        plan_rows.append({
            'Day'      : f'Day {day_idx + 1}',
            'Day_Num'  : day_idx + 1,
            'Meal'     : meal_type,
            'Recipe'   : row['Recipe_Name'],
            'Calories' : round(row['Calories'],  1),
            'Carbs_g'  : round(row['Carbs_g'],   1),
            'Sugar_g'  : round(row['Sugar_g'],   1),
            'Protein_g': round(row['Protein_g'], 1),
            'RF_Score' : round(row['RF_Score'],  4),
        })

meal_order     = {'Breakfast':0, 'Lunch':1, 'Dinner':2}
plan_df        = pd.DataFrame(plan_rows)
plan_df['Meal_Order'] = plan_df['Meal'].map(meal_order)
plan_df        = (
    plan_df
    .sort_values(['Day_Num', 'Meal_Order'])
    .drop(columns=['Day_Num', 'Meal_Order'])
    .reset_index(drop=True)
)

print('30-Day Random Forest Diet Plan Generated')
print('=' * 55)
print(f'  Total meals : {len(plan_df)} (30 days x 3 slots)')
for mt, c in pool_summary.items():
    print(f'  {mt:12s}: {c} unique meals by MMR')
print()
print('Sample — First 9 meals:')
print(plan_df[['Day','Meal','Recipe',
               'Calories','Carbs_g','Sugar_g','RF_Score']]
      .head(9).to_string(index=False))

30-Day Random Forest Diet Plan Generated
  Total meals : 90 (30 days x 3 slots)
  Breakfast   : 30 unique meals by MMR
  Lunch       : 30 unique meals by MMR
  Dinner      : 30 unique meals by MMR

Sample — First 9 meals:
  Day      Meal                              Recipe  Calories  Carbs_g  Sugar_g  RF_Score
Day 1 Breakfast      wisconsin beer batter for fish     478.0     44.0      0.0    0.6969
Day 1     Lunch           broccoli rice and chicken     482.7     41.2      0.5    0.7077
Day 1    Dinner mark bittman s crunchy curried fish     390.6     44.0      0.5    0.6774
Day 2 Breakfast                  crispy fish sticks     448.0     52.2      7.0    0.6255
Day 2     Lunch               couscous with chicken     469.4     46.8      8.0    0.6257
Day 2    Dinner   hot roast beef sandwiches   gravy     666.6     44.0      2.5    0.6513
Day 3 Breakfast             parmesan chicken strips     513.1     41.2      4.0    0.6701
Day 3     Lunch                  himalayan red rice     23

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 12 — PLAN EVALUATION
# ════════════════════════════════════════════════════════

print('=' * 55)
print('   RANDOM FOREST PLAN EVALUATION')
print('=' * 55)

compliant = plan_df[
    (plan_df['Carbs_g']  <= patient_macros['per_meal_max_carbs']) &
    (plan_df['Carbs_g']  >= patient_macros['per_meal_min_carbs']) &
    (plan_df['Sugar_g']  <= patient_macros['per_meal_sugar'])     &
    (plan_df['Calories'] <= patient_macros['per_meal_calories'])
]

print(f'\n1. Medical Compliance')
print(f'   ADA-safe meals  : {len(compliant)} / {len(plan_df)}')
print(f'   Rate            : {len(compliant)/len(plan_df)*100:.1f}%')

print(f'\n2. Nutritional Profile')
print(f'   Avg Calories    : {plan_df["Calories"].mean():.1f} kcal')
print(f'   Avg Carbs       : {plan_df["Carbs_g"].mean():.1f}g '
      f'(ADA target 45-60g)')
print(f'   Avg Sugar       : {plan_df["Sugar_g"].mean():.1f}g')
print(f'   Avg Protein     : {plan_df["Protein_g"].mean():.1f}g')
print(f'   Avg RF Score    : {plan_df["RF_Score"].mean():.4f}')

unique = plan_df['Recipe'].nunique()
print(f'\n3. Variety')
print(f'   Unique recipes  : {unique} / {len(plan_df)}')
print(f'   Variety score   : {unique/len(plan_df)*100:.1f}%')

print(f'\n4. Per-Slot Variety')
for mt in MEAL_SLOTS:
    u = plan_df[plan_df['Meal']==mt]['Recipe'].nunique()
    print(f'   {mt:12s}: {u} unique / 30 days')

in_ada = plan_df[
    (plan_df['Carbs_g'] >= 45) & (plan_df['Carbs_g'] <= 60)
]
print(f'\n5. ADA Carb Range (45-60g)')
print(f'   In range: {len(in_ada)} / {len(plan_df)} meals')
print('\n' + '=' * 55)

   RANDOM FOREST PLAN EVALUATION

1. Medical Compliance
   ADA-safe meals  : 90 / 90
   Rate            : 100.0%

2. Nutritional Profile
   Avg Calories    : 514.8 kcal
   Avg Carbs       : 42.8g (ADA target 45-60g)
   Avg Sugar       : 2.4g
   Avg Protein     : 45.8g
   Avg RF Score    : 0.6558

3. Variety
   Unique recipes  : 90 / 90
   Variety score   : 100.0%

4. Per-Slot Variety
   Breakfast   : 30 unique / 30 days
   Lunch       : 30 unique / 30 days
   Dinner      : 30 unique / 30 days

5. ADA Carb Range (45-60g)
   In range: 32 / 90 meals



In [ ]:
# ════════════════════════════════════════════════════════
# CELL 13 — PDF REPORT
# ════════════════════════════════════════════════════════

from fpdf import FPDF

def to_safe(text, limit=38):
    return str(text).encode(
        'latin-1', errors='replace'
    ).decode('latin-1')[:limit].title()


class RfPDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 13)
        self.set_text_color(29, 158, 117)
        self.cell(0, 10, 'DIAPILOT - Random Forest Diet Plan',
                  ln=True, align='C')
        self.set_font('Arial', 'I', 9)
        self.set_text_color(100, 100, 100)
        self.cell(0, 6,
                  'Experiment 4: Random Forest Regressor | '
                  'Feature Importance + MMR | ADA 2025',
                  ln=True, align='C')
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 10,
                  f'Page {self.page_no()} | '
                  f'Diapilot Random Forest Experiment 4',
                  align='C')


pdf = RfPDF()
pdf.set_auto_page_break(auto=True, margin=20)
pdf.add_page()

# Patient box
pdf.set_font('Arial', 'B', 9)
pdf.set_fill_color(225, 245, 238)
pdf.set_text_color(8, 80, 65)
pdf.cell(0, 8,
         f'Patient | TDEE: {patient_tdee} kcal | '
         f'Leicester: {LEICESTER_SCORE} | '
         f'Medication: {"Yes" if ON_MEDICATION else "No"} | '
         f'Carbs: {patient_macros["per_meal_min_carbs"]}'
         f'-{patient_macros["per_meal_max_carbs"]}g | '
         f'Sugar: max {patient_macros["per_meal_sugar"]}g | '
         f'Trees: 300',
         border=1, ln=True, fill=True, align='C')
pdf.ln(3)

# Feature importance mini-table
pdf.set_font('Arial', 'B', 9)
pdf.set_fill_color(45, 80, 65)
pdf.set_text_color(255, 255, 255)
pdf.cell(130, 7, 'Top Feature Importances (RF Unique Insight)',
         border=1, align='C', fill=True)
pdf.cell(60,  7, 'Importance %',
         border=1, align='C', fill=True)
pdf.ln()

pdf.set_font('Arial', '', 8)
for _, row in importance_df.head(5).iterrows():
    fill = row['Type'] == 'Patient'
    pdf.set_fill_color(240, 248, 255) if fill else \
        pdf.set_fill_color(255, 253, 240)
    pdf.set_text_color(30, 30, 30)
    pdf.cell(130, 6,
             f'  [{row["Type"]}] {row["Feature"]}',
             border=1, fill=True)
    pdf.cell(60,  6,
             f'{row["Importance_Pct"]}%',
             border=1, align='C', fill=True)
    pdf.ln()
pdf.ln(4)

# Main table header
col_widths = [18, 25, 70, 20, 18, 18, 16, 20]
headers    = ['Day','Meal','Recipe Name','Cal',
              'Carbs','Sugar','Pro','RF Score']
pdf.set_font('Arial', 'B', 9)
pdf.set_fill_color(29, 158, 117)
pdf.set_text_color(255, 255, 255)
for h, w in zip(headers, col_widths):
    pdf.cell(w, 8, h, border=1, align='C', fill=True)
pdf.ln()

meal_colors = {
    'Breakfast': (255, 253, 240),
    'Lunch'    : (240, 248, 255),
    'Dinner'   : (245, 240, 255),
}
pdf.set_font('Arial', '', 8)
current_day = ''

for _, row in plan_df.iterrows():
    r, g, b     = meal_colors[row['Meal']]
    pdf.set_fill_color(r, g, b)
    pdf.set_text_color(30, 30, 30)
    day_label   = row['Day'] if row['Day'] != current_day else ''
    current_day = row['Day']

    pdf.cell(col_widths[0], 7, day_label,
             border=1, align='C', fill=True)
    pdf.cell(col_widths[1], 7, row['Meal'],
             border=1, align='C', fill=True)
    pdf.cell(col_widths[2], 7, to_safe(row['Recipe']),
             border=1, fill=True)
    pdf.cell(col_widths[3], 7, str(row['Calories']),
             border=1, align='C', fill=True)
    pdf.cell(col_widths[4], 7, f"{row['Carbs_g']}g",
             border=1, align='C', fill=True)
    pdf.cell(col_widths[5], 7, f"{row['Sugar_g']}g",
             border=1, align='C', fill=True)
    pdf.cell(col_widths[6], 7, f"{row['Protein_g']}g",
             border=1, align='C', fill=True)
    pdf.cell(col_widths[7], 7, f"{row['RF_Score']:.3f}",
             border=1, align='C', fill=True)
    pdf.ln()

os.makedirs('../reports', exist_ok=True)
pdf.output('../reports/Diapilot_RF_Diet_Plan.pdf')
plan_df.to_csv('../reports/Diapilot_RF_Diet_Plan.csv', index=False)
joblib.dump(rf_model,      '../models/diapilot_rf_model.pkl')
joblib.dump(importance_df, '../models/diapilot_rf_importance.pkl')

print('PDF saved  : ../reports/Diapilot_RF_Diet_Plan.pdf')
print('CSV saved  : ../reports/Diapilot_RF_Diet_Plan.csv')
print('Model saved: ../models/diapilot_rf_model.pkl')
print(f'Total meals: {len(plan_df)}')

PDF saved  : ../reports/Diapilot_RF_Diet_Plan.pdf
CSV saved  : ../reports/Diapilot_RF_Diet_Plan.csv
Model saved: ../models/diapilot_rf_model.pkl
Total meals: 90
